# Profundidades con TGLC: ¿la inconsistencia entre sectores viene de los datos?

**Kernel: `starry`.** Requiere haber corrido `1c_tglc_extraccion.ipynb` (kernel `tglc`).

La sección 7 de `1b_profundidades_eclipse_v2.ipynb` marca varios sistemas cuyas profundidades
cambian de un sector a otro. Si la causa es la fotometría (dilución por vecinas o por la nebulosa,
fondo mal restado, apertura distinta en cada sector), una fotometría independiente que modela y resta
las vecinas debería dar otro resultado. Si la causa es real (manchas, cambio de geometría), TGLC
debería ver el mismo cambio.

Aquí se repite **exactamente la misma medición** del 1b sobre las curvas TGLC:
- mismas funciones (`normalizar_sin_eclipses`, `medir`, `consistencia`, `figura_plegadas`), cargadas
  directamente del notebook 1b para no duplicar código;
- mismas efemérides por sector ($P_{\rm orb}$, $t_0$, duración) de `priors_v2.csv`, así que la única
  diferencia entre las dos mediciones es la fotometría.

**Cómo leer el resultado:**

| propia | TGLC | interpretación |
|---|---|---|
| difiere entre sectores | consistente | problema de los datos propios (contaminación que cambia por sector) |
| difiere | difiere igual (mismos sectores, mismo sentido) | efecto real o de la efeméride |
| D distinta a TGLC en el mismo sector, D2/D1 igual | — | dilución en la fotometría propia |

In [ ]:
import json
import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUTA_ORION = Path("/home/luciam/Documents/starry/Resultados_Orion")
NB_1B = RUTA_ORION / "1b_profundidades_eclipse_v2.ipynb"


def cargar_de_1b(celdas, nombres=()):
    # ejecuta de las celdas del 1b solo imports, definiciones de funciones y las asignaciones en `nombres`
    nb = json.load(open(NB_1B))
    for i in celdas:
        arbol = ast.parse("".join(nb["cells"][i]["source"]))
        nodos = [n for n in arbol.body if isinstance(n, (ast.Import, ast.ImportFrom, ast.FunctionDef))
                 or (isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and t.id in nombres
                                                       for t in n.targets))
                 or (isinstance(n, ast.Assign) and any(isinstance(t, ast.Tuple) and
                                                       any(e.id in nombres for e in t.elts) for t in n.targets))]
        exec(compile(ast.Module(body=nodos, type_ignores=[]), f"1b[{i}]", "exec"), globals())


cargar_de_1b([4], ("RUTA_CURVAS", "WINDOW_WOTAN", "ALFAS", "N_MIN", "W_MAX", "TOL_P"))
cargar_de_1b([5], ("ALFA_0",))
cargar_de_1b([17], ("P_UMBRAL", "F_UMBRAL", "CAD_COLOR", "csv_idx", "INDICE"))
cargar_de_1b([18])
cargar_de_1b([20], ("COLOR_SECTOR",))

DIR_TXT = RUTA_ORION / "Light_Curve_Files_TGLC" / "local"
DIR_FIG = RUTA_ORION / "resultados_pipeline" / "profundidades_v2" / "tglc"
DIR_FIG.mkdir(parents=True, exist_ok=True)

# fotometría TGLC a usar. cal_psf_flux: ajuste de PSF (mejor en campos poblados y estrellas débiles);
# cal_aper_flux: apertura 3x3 sobre la imagen descontaminada (mejor en estrellas brillantes)
FLUJO_TGLC = "cal_psf_flux"
print("funciones del 1b:", [f.__name__ for f in (medir, consistencia, figura_plegadas, figura_sectores)])

## 1) Curvas

`cargar_tglc` aplica las banderas de calidad (`TGLC_flags == 0` y `TESS_flags == 0`). Se miden las dos
fotometrías: la propia (`Flux_CBV_SS`, con `cargar_curva` del 1b) y la TGLC elegida.

In [ ]:
def cargar_tglc(tic, sector, columna=FLUJO_TGLC):
    p = DIR_TXT / f"tic {tic}_S_{sector}_TGLC.txt"
    if not p.exists():
        return None
    d = pd.read_csv(p, sep=" ", comment="#")
    d = d.loc[(d["TGLC_flags"] == 0) & (d["TESS_flags"] == 0)].dropna(subset=["time", columna])
    d = d.loc[d[columna] > 0].sort_values("time").drop_duplicates("time")
    return d["time"].to_numpy(float), d[columna].to_numpy(float)


v2 = pd.read_csv(RUTA_ORION / "priors_v2.csv")
disp = sorted({(int(p.name.split()[1].split("_")[0]), int(p.name.split("_S_")[1].split("_")[0]))
               for p in DIR_TXT.glob("tic *_S_*_TGLC.txt")})
tics = sorted({t for t, _ in disp})
print(f"{len(disp)} curvas TGLC de {len(tics)} TIC")
pd.DataFrame(disp, columns=["TIC", "Sector"]).groupby("TIC")["Sector"].apply(list)

## 2) Medición con la efeméride de `priors_v2.csv`

Para cada (TIC, sector) se llama `medir(t, f, P_orb, t0, Duration)` con las dos fotometrías. Las filas
quedan con las mismas columnas que la tabla `med` del 1b, de modo que las funciones de figura del 1b
funcionan sin cambios. Un sector sin curva TGLC queda fuera de `med_tglc` (pero sí está en `med_propia`).

In [ ]:
def medir_todo(cargador):
    filas, curvas_ = [], {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for r in v2.loc[v2["TIC"].isin(tics)].itertuples():
            tic, s = int(r.TIC), int(r.Sector)
            if (tic, s) not in disp:
                continue
            tf = cargador(tic, s)
            if tf is None or len(tf[0]) < 100:
                continue
            t, f = tf
            m, _, _ = medir(t, f, r.P_orb, r.t0, r.Duration)
            curvas_[(tic, s)] = (t, f)
            filas.append(dict(TIC=tic, Sector=s, P=r.P_orb, t0=r.t0, Duration=r.Duration,
                              texp_sobre_duracion=np.median(np.diff(t)) / r.Duration, **m))
    return pd.DataFrame(filas), curvas_


med_propia, curvas_propia = medir_todo(cargar_curva)
med_tglc, curvas_tglc = medir_todo(cargar_tglc)
print(f"propia: {len(med_propia)} curvas · TGLC ({FLUJO_TGLC}): {len(med_tglc)} curvas")

## 3) Comparación sector a sector: propia vs TGLC

- `D1_tglc / D1_propia`: si es > 1 de forma sistemática, la curva propia está diluida (tiene flujo de
  más). La fracción de flujo extra en la propia es $1 - D_{\rm propia}/D_{\rm TGLC}$.
- `D2/D1`: no depende de la dilución. Si coincide entre fotometrías, la diferencia en D es solo dilución;
  si no, alguna de las dos curvas tiene un problema de forma (fondo variable, eclipse mal muestreado).

In [ ]:
cols = ["D_primario", "D_primario_err", "D_secundario", "D_secundario_err", "secundario_detectado"]
comp = med_propia[["TIC", "Sector"] + cols].merge(med_tglc[["TIC", "Sector"] + cols], on=["TIC", "Sector"],
                                                  suffixes=("_propia", "_tglc"))
comp["D1_tglc/propia"] = comp["D_primario_tglc"] / comp["D_primario_propia"]
comp["D2_tglc/propia"] = comp["D_secundario_tglc"] / comp["D_secundario_propia"]
comp["D2/D1_propia"] = comp["D_secundario_propia"] / comp["D_primario_propia"]
comp["D2/D1_tglc"] = comp["D_secundario_tglc"] / comp["D_primario_tglc"]
comp["flujo_extra_propia"] = 1 - 1 / comp["D1_tglc/propia"]
comp.to_csv(DIR_FIG / f"comparacion_propia_vs_tglc_{FLUJO_TGLC}.csv", index=False)
comp[["TIC", "Sector", "D_primario_propia", "D_primario_tglc", "D1_tglc/propia", "D2_tglc/propia",
      "D2/D1_propia", "D2/D1_tglc", "flujo_extra_propia"]].round(4)

In [ ]:
# figura resumen: cociente de profundidades por sector (dilución) y D2/D1 en ambas fotometrías
fig, axs = plt.subplots(1, 2, figsize=(13, 0.42 * len(tics) + 2), sharey=True)
y = {tic: k for k, tic in enumerate(tics)}
for _, r in comp.iterrows():
    col = COLOR_SECTOR.get(int(r["Sector"]), "0.4")
    yy = y[r["TIC"]] + {5: -0.2, 6: -0.2, 32: -0.07, 33: -0.07, 87: 0.07, 98: 0.2}.get(int(r["Sector"]), 0)
    axs[0].plot(r["D1_tglc/propia"], yy, "o", color=col, mec="0.2")
    axs[0].plot(r["D2_tglc/propia"], yy, "s", color=col, mec="0.2", ms=5, alpha=0.7)
    axs[1].plot(r["D2/D1_propia"], yy, "o", color=col, mec="0.2")
    axs[1].plot(r["D2/D1_tglc"], yy, "x", color="k", ms=7)
    axs[1].plot([r["D2/D1_propia"], r["D2/D1_tglc"]], [yy, yy], "-", color=col, lw=1)
axs[0].axvline(1, color="0.5", lw=1)
axs[0].set_xlabel("D TGLC / D propia   (● primario, ■ secundario)")
axs[1].set_xlabel("D2/D1   (● propia, × TGLC)")
axs[1].set_xscale("log")
axs[0].set_yticks(range(len(tics)))
axs[0].set_yticklabels([(f"sist. {int(INDICE[t])} · " if t in INDICE else "") + f"TIC {t}" for t in tics], fontsize=8)
for ax in axs:
    ax.grid(alpha=0.25)
from matplotlib.lines import Line2D
axs[0].legend(handles=[Line2D([], [], ls="", marker="o", color=c, label=f"S{s}") for s, c in
                       ((6, "#2a78d6"), (32, "#eb6834"), (87, "#eda100"), (98, "#1baf7a"))],
              fontsize=8, loc="lower right")
fig.suptitle(f"Fotometría propia vs TGLC ({FLUJO_TGLC}), mismo sector y misma efeméride", y=1.0)
fig.tight_layout()
fig.savefig(DIR_FIG / "propia_vs_tglc_cocientes.png", dpi=130, bbox_inches="tight")
plt.show()

## 4) Consistencia entre sectores con TGLC (sección 7 del 1b)

Misma prueba χ² y mismas marcas (`consistente`, `difiere_leve`, `difiere_revisar`) que en el 1b, ahora
con las profundidades TGLC. La tabla final pone lado a lado la marca de cada fotometría.

In [ ]:
def tabla_consistencia(med):
    filas_cons = []
    for tic, g in med.groupby("TIC"):
        g = g.loc[~g["eclipse_ancho"]].sort_values("Sector")
        if g.empty:
            continue
        r1 = consistencia(g["D_primario"], g["D_primario_err"], g["alfa_primario"])
        g2 = g.loc[g["secundario_detectado"].astype(bool)]
        r2 = consistencia(g2["D_secundario"], g2["D_secundario_err"], g2["alfa_secundario"])
        cociente = (g2["D_secundario"] / g2["D_primario"]).values
        filas_cons.append(dict(
            TIC=tic, indice=INDICE.get(tic, np.nan), sectores=",".join(str(int(s)) for s in g["Sector"]),
            D1_media=r1["media"], D1_media_err=r1["media_err"], D1_chi2red=r1["chi2red"], D1_p=r1["p"],
            D1_f_extra=r1["f_extra"], marca_primario=r1["marca"],
            D2_media=r2["media"], D2_chi2red=r2["chi2red"], D2_p=r2["p"], D2_f_extra=r2["f_extra"],
            marca_secundario=r2["marca"] if r2["n"] else "sin_secundario",
            D2_D1_rango=(np.ptp(cociente) / np.mean(cociente)) if len(cociente) > 1 else np.nan))
    c = pd.DataFrame(filas_cons)
    c["causa_probable"] = c.apply(causa, axis=1)
    return c


cons_propia = tabla_consistencia(med_propia)
cons_tglc = tabla_consistencia(med_tglc)
cons_tglc.to_csv(DIR_FIG / f"consistencia_sectores_tglc_{FLUJO_TGLC}.csv", index=False)

lado = cons_propia[["TIC", "indice", "sectores", "D1_media", "D1_chi2red", "marca_primario", "marca_secundario",
                    "D2_D1_rango"]].merge(
    cons_tglc[["TIC", "sectores", "D1_media", "D1_chi2red", "marca_primario", "marca_secundario", "D2_D1_rango"]],
    on="TIC", suffixes=("_propia", "_tglc"))


def veredicto(r):
    if r["sectores_propia"] != r["sectores_tglc"]:
        return "faltan sectores TGLC: comparar con cuidado"
    if "un_sector" in (r["marca_primario_propia"], r["marca_primario_tglc"]):
        return "un solo sector: sin prueba"
    p_ok, t_ok = r["marca_primario_propia"] == "consistente", r["marca_primario_tglc"] == "consistente"
    if not p_ok and t_ok:
        return "problema de la fotometría propia"
    if not p_ok and not t_ok:
        return "TGLC también difiere: real, efeméride o ambas fotometrías"
    if p_ok and not t_ok:
        return "solo TGLC difiere: revisar curva TGLC"
    return "ambas consistentes"


lado["veredicto"] = lado.apply(veredicto, axis=1)
lado.sort_values("indice").round(4)

## 5) Figura de curvas plegadas por sector (como `plegadas_sectores_todos.png`)

Se usa `figura_plegadas` del 1b tal cual; solo se le cambian las tablas globales `med`, `cons` y `curvas`
por las de TGLC. Se genera también la versión propia con exactamente los mismos sistemas, para poner
las dos figuras lado a lado.

In [ ]:
def figura_todos(med_, cons_, curvas_, nombre_archivo, titulo):
    global med, cons, curvas
    med, cons, curvas = med_, cons_, curvas_
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        orden = cons.sort_values("indice")["TIC"].tolist()
        nfil = int(np.ceil(len(orden) / 2))
        fig, axs = plt.subplots(nfil, 4, figsize=(20, 3.9 * nfil), squeeze=False,
                                gridspec_kw={"hspace": 0.85, "wspace": 0.28})
        for k, tic in enumerate(orden):
            figura_plegadas(tic, axs=axs[k // 2][2 * (k % 2): 2 * (k % 2) + 2], guardar=False)
        for k in range(len(orden), 2 * nfil):
            for ax in axs[k // 2][2 * (k % 2): 2 * (k % 2) + 2]:
                ax.axis("off")
        fig.subplots_adjust(top=1 - 1.0 / fig.get_figheight())
        fig.suptitle(titulo, y=1 - 0.3 / fig.get_figheight(), fontsize=13)
        fig.savefig(DIR_FIG / nombre_archivo, dpi=110, bbox_inches="tight")
        plt.show()


figura_todos(med_tglc, cons_tglc, curvas_tglc, f"plegadas_sectores_tglc_{FLUJO_TGLC}.png",
             f"TGLC ({FLUJO_TGLC}): eclipses plegados por sector (línea discontinua = 1 − D de cada sector)")

In [ ]:
figura_todos(med_propia, cons_propia, curvas_propia, "plegadas_sectores_propia_mismos_tic.png",
             "Fotometría propia (Flux_CBV_SS), mismos sistemas: eclipses plegados por sector")

## 6) Profundidad por sector, propia vs TGLC

Como `consistencia_sectores_todos.png` del 1b, con una fila por sistema: a la izquierda la fotometría
propia y a la derecha TGLC (`figura_sectores` del 1b).

In [ ]:
orden = cons_propia.sort_values("indice")["TIC"].tolist()
fig, axs = plt.subplots(len(orden), 2, figsize=(10, 4.0 * len(orden)), squeeze=False,
                        gridspec_kw={"hspace": 0.8, "wspace": 0.35})
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for fila_ax, tic in zip(axs, orden):
        for ax, (med, cons) in zip(fila_ax, ((med_propia, cons_propia), (med_tglc, cons_tglc))):
            if (cons["TIC"] == tic).any():
                figura_sectores(tic, ax=ax, guardar=False)
            else:
                ax.axis("off")
axs[0][0].set_title("propia\n\n\n", fontsize=11)
axs[0][1].set_title(f"TGLC ({FLUJO_TGLC})\n\n\n", fontsize=11)
fig.savefig(DIR_FIG / f"consistencia_sectores_propia_vs_tglc_{FLUJO_TGLC}.png", dpi=110, bbox_inches="tight")
plt.show()

## 7) Validación: TGLC local vs HLSP de MAST (S5/S6)

Las curvas HLSP de MAST (`Light_Curve_Files_TGLC/*.fits`) y las locales deberían dar casi la misma
profundidad en S5/S6. Una diferencia grande indica un problema de la corrida local (tamaño del corte,
catálogo Gaia, versión de TGLC).

In [ ]:
from astropy.io import fits

filas = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for p in sorted((RUTA_ORION / "Light_Curve_Files_TGLC").glob("*.fits")):
        with fits.open(p) as h:
            tic, s, d = int(h[0].header["TICID"]), int(h[0].header["SECTOR"]), h[1].data
        sel = med_tglc.loc[(med_tglc["TIC"] == tic) & (med_tglc["Sector"] == s)]
        r = v2.loc[(v2["TIC"] == tic) & (v2["Sector"] == s)]
        if sel.empty or r.empty:
            continue
        ok = (d["TGLC_flags"] == 0) & (d["TESS_flags"] == 0) & np.isfinite(d[FLUJO_TGLC]) & (d[FLUJO_TGLC] > 0)
        r = r.iloc[0]
        m, _, _ = medir(d["time"][ok].astype(float), d[FLUJO_TGLC][ok].astype(float), r["P_orb"], r["t0"],
                        r["Duration"])
        filas.append(dict(TIC=tic, Sector=s, gaia=p.name.split("-")[1], D1_mast=m["D_primario"],
                          D1_local=sel["D_primario"].iloc[0], D2_mast=m["D_secundario"],
                          D2_local=sel["D_secundario"].iloc[0]))
val = pd.DataFrame(filas)
if len(val):
    val["D1_local/mast"] = val["D1_local"] / val["D1_mast"]
val.round(4)

## 8) ¿Variación real de la estrella o efecto sistemático?

Una profundidad promedio por sector no distingue las dos causas. Aquí se mide **cada eclipse por
separado** y se buscan las firmas de cada una.

**Profundidad de cada eclipse.** Con el flujo normalizado de §2.1 y el nivel de referencia $B$ del sector,
para cada evento se busca el mínimo de la mediana móvil (ventana de $0.3\,W$ pero nunca menor que dos
cadencias, al menos 2 puntos) con el
centro libre dentro de $\pm W/8$ de la efeméride. Así cada eclipse se mide en su propio mínimo y **una
efeméride imprecisa no achata la profundidad** (el problema de S98). El error es
$1.2533\,\sigma/\sqrt{n}$, igual que en §2.3. Se guarda también el **nivel de brillo fuera de eclipse** en
ese momento: la tendencia biweight $T(t)$ dividida por su mediana en el sector.

**Firmas:**

| observación | significado |
|---|---|
| eclipses de un mismo sector coinciden entre sí, pero los sectores difieren entre sí (ANOVA) | algo constante dentro del sector: dilución/apertura (sistemático) o un cambio lento real |
| propia difiere entre sectores y TGLC no | sistemático de la fotometría propia |
| el cociente $D_{\rm propia}/D_{\rm TGLC}$ cambia de sector a sector | la contaminación de la fotometría propia cambia por sector |
| profundidad de cada eclipse correlacionada con el brillo fuera de eclipse (dentro del sector) | manchas: variación **real** del flujo de la estrella |
| dispersión entre eclipses mayor que el ruido (χ²/ν ≫ 1) sin correlación con el brillo | otra variabilidad real (acreción, discos, flares) o ruido subestimado |
| D2/D1 cambia | real (manchas en una de las estrellas) o problema de forma; la dilución no cambia D2/D1 |

Para manchas no ocultadas en la estrella eclipsada, cuando la estrella está más brillante (menos manchas
visibles) el eclipse es relativamente más somero, así que se espera una correlación **negativa** entre D
y el nivel de brillo.

In [ ]:
from scipy.stats import spearmanr, f as dist_f


def eclipses_individuales(t, f, r):
    # r: fila de med_* (P, t0, W, B, centro_primario, fase_secundario, secundario_detectado)
    P, W, B = r["P"], r["W"], r["B"]
    phi2 = r["fase_secundario"] if r["secundario_detectado"] else 0.5
    F, T = normalizar_sin_eclipses(t, f, r["t0"], P, W, phi2)
    sigma = ruido_por_punto(F)
    nivel = T / np.nanmedian(T)
    cad = np.median(np.diff(t)) / P                 # cadencia en unidades de fase
    h = max(ALFA_0 * W / 2, 1.01 * cad)             # ventana nunca menor que una cadencia (S6: 30 min)
    nucleo = max(W / 4, 1.01 * cad)
    filas = []
    for nombre, c in (("primario", r["centro_primario"] if np.isfinite(r["centro_primario"]) else 0.0),
                      ("secundario", phi2)):
        if nombre == "secundario" and not r["secundario_detectado"]:
            continue
        t_c0 = r["t0"] + c * P
        x = (t - t_c0) / P
        ciclo = np.round(x)
        ph = x - ciclo
        for n in np.unique(ciclo[np.abs(ph) < W / 2]):
            sel = (ciclo == n) & (np.abs(ph) < W)
            p_, F_ = ph[sel], F[sel]
            # cobertura: puntos en el núcleo y en ambos flancos del eclipse
            if (np.abs(p_) < nucleo).sum() < 2 or not ((p_ < -W / 4).any() and (p_ > W / 4).any()):
                continue
            centros = np.arange(-W / 8, W / 8 + 1e-12, max(h / 4, 1e-5))
            meds, ns = medianas_ventana(p_, F_, centros, h, 2)
            if np.all(np.isnan(meds)):
                continue
            j = int(np.nanargmin(meds))
            filas.append(dict(eclipse=nombre, ciclo=int(n), t=t_c0 + (n + centros[j]) * P, D=B - meds[j],
                              err=1.2533 * sigma / np.sqrt(ns[j]), n=int(ns[j]),
                              nivel=float(np.interp(t_c0 + (n + centros[j]) * P, t, nivel))))
    return filas


def tabla_individuales(med, curvas_, fotometria):
    filas = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for _, r in med.loc[~med["eclipse_ancho"]].iterrows():
            t, f = curvas_[(r["TIC"], r["Sector"])]
            for e in eclipses_individuales(t, f, r):
                filas.append(dict(TIC=r["TIC"], Sector=r["Sector"], fotometria=fotometria, **e))
    return pd.DataFrame(filas)


ind = pd.concat([tabla_individuales(med_propia, curvas_propia, "propia"),
                 tabla_individuales(med_tglc, curvas_tglc, "tglc")], ignore_index=True)
ind.to_csv(DIR_FIG / f"eclipses_individuales_{FLUJO_TGLC}.csv", index=False)
ind.groupby(["TIC", "fotometria", "eclipse", "Sector"])["D"].agg(["count", "median", "std"]).round(4)

### 8.1 Estadísticos por sistema, fotometría y eclipse

- `chi2_intra`: χ²/ν de los eclipses alrededor de la media de su sector (≈ 1 si solo hay ruido).
- `p_entre`: ANOVA de una vía; probabilidad de que las medias de los sectores difieran solo por la
  dispersión entre eclipses. Pequeño = hay un salto de sector a sector que la dispersión eclipse a eclipse
  no explica.
- `rho_brillo`, `p_brillo`: Spearman entre la profundidad y el nivel de brillo, **dentro de cada sector**
  (se restan la media de D y del nivel de cada sector antes de juntar).
- `D_med_S<s>`: mediana de las profundidades individuales por sector (libre del achatamiento por
  efeméride).

In [ ]:
def estadisticos(g):
    out = {}
    secs = sorted(g["Sector"].unique())
    grupos = [g.loc[g["Sector"] == s] for s in secs]
    grupos = [x for x in grupos if len(x) >= 2]
    for s in secs:
        out[f"D_med_S{s}"] = g.loc[g["Sector"] == s, "D"].median()
    if grupos:
        res = np.concatenate([(x["D"] - np.average(x["D"], weights=x["err"] ** -2)) / x["err"] for x in grupos])
        nu = len(res) - len(grupos)
        out["chi2_intra"] = float(np.sum(res ** 2) / nu) if nu > 0 else np.nan
        dD = np.concatenate([x["D"] - x["D"].mean() for x in grupos])
        dN = np.concatenate([x["nivel"] - x["nivel"].mean() for x in grupos])
        if len(dD) >= 6 and np.ptp(dN) > 0:
            out["rho_brillo"], out["p_brillo"] = spearmanr(dN, dD)
    if len(grupos) >= 2:
        k, N = len(grupos), sum(len(x) for x in grupos)
        gm = np.mean(np.concatenate([x["D"] for x in grupos]))
        ss_b = sum(len(x) * (x["D"].mean() - gm) ** 2 for x in grupos)
        ss_w = sum(((x["D"] - x["D"].mean()) ** 2).sum() for x in grupos)
        if ss_w > 0 and N > k:
            out["F_entre"] = (ss_b / (k - 1)) / (ss_w / (N - k))
            out["p_entre"] = float(dist_f.sf(out["F_entre"], k - 1, N - k))
    out["n_eclipses"] = len(g)
    return out


est = pd.DataFrame([dict(TIC=k[0], fotometria=k[1], eclipse=k[2], **estadisticos(g))
                    for k, g in ind.groupby(["TIC", "fotometria", "eclipse"])])
est.to_csv(DIR_FIG / f"estadisticos_eclipses_individuales_{FLUJO_TGLC}.csv", index=False)
est.round(4)

### 8.2 Veredicto por sistema

Reglas (umbrales: $p < 0.0027$, 3σ, igual que en el 1b), aplicadas al eclipse primario:

1. **manchas (real)**: correlación significativa entre D y brillo dentro de los sectores, en ambas
   fotometrías (o en la única disponible).
2. **sistemático de la fotometría propia**: los sectores difieren en la propia (`p_entre` pequeño) y no en
   TGLC; o el cociente propia/TGLC de las medianas por sector varía más de 5 %.
3. **salto entre sectores en ambas fotometrías**: sin correlación con el brillo. Candidatos: luz de una
   compañera no resuelta por Gaia (que sí puede variar), fondo nebular, o un cambio lento real (geometría,
   actividad). Mirar si D2/D1 también cambia y si la duración cambia.
4. **consistente**: sin salto entre sectores en la fotometría propia.

El veredicto es una guía; las figuras de abajo son las que deciden.

In [ ]:
def veredicto_real(tic):
    e = est.loc[(est["TIC"] == tic) & (est["eclipse"] == "primario")].set_index("fotometria")
    if "propia" not in e.index:
        return pd.Series(dict(veredicto="sin datos"))
    pr = e.loc["propia"]
    tg = e.loc["tglc"] if "tglc" in e.index else None
    notas = []
    salto_p = pr.get("p_entre", np.nan) < P_UMBRAL
    salto_t = tg is not None and tg.get("p_entre", np.nan) < P_UMBRAL
    manchas = [x.get("p_brillo", np.nan) < P_UMBRAL for x in (pr, tg) if x is not None]
    secs = [c for c in e.columns if c.startswith("D_med_S")]
    cociente = (pr[secs] / tg[secs]).astype(float).dropna() if tg is not None else pd.Series(dtype=float)
    var_cociente = np.ptp(cociente) / cociente.mean() if len(cociente) >= 2 else np.nan
    if len(cociente):
        notas.append("D_propia/D_TGLC por sector: " + ", ".join(f"{k[6:]}={v:.2f}" for k, v in cociente.items()))
    if manchas and all(manchas):
        v = "variación real (manchas): D correlaciona con el brillo"
    elif salto_p and tg is not None and not salto_t and np.isfinite(tg.get("p_entre", np.nan)):
        v = "sistemático de la fotometría propia (TGLC consistente entre sectores)"
    elif np.isfinite(var_cociente) and var_cociente > 0.05:
        v = "sistemático: la contaminación de la fotometría propia cambia por sector"
    elif salto_p and salto_t:
        v = "salto entre sectores en ambas fotometrías: compañera no resuelta, fondo o cambio lento real"
    elif salto_p:
        v = "salto entre sectores en la propia (sin TGLC suficiente para comparar)"
    elif not np.isfinite(pr.get("p_entre", np.nan)):
        v = "un solo sector con eclipses medibles: sin prueba entre sectores"
    else:
        v = "consistente"
    if any(manchas) and not all(manchas):
        notas.append("correlación con el brillo solo en una fotometría")
    return pd.Series(dict(veredicto=v, p_entre_propia=pr.get("p_entre", np.nan),
                          p_entre_tglc=tg.get("p_entre", np.nan) if tg is not None else np.nan,
                          rho_brillo_propia=pr.get("rho_brillo", np.nan),
                          rho_brillo_tglc=tg.get("rho_brillo", np.nan) if tg is not None else np.nan,
                          chi2_intra_propia=pr.get("chi2_intra", np.nan),
                          variacion_cociente=var_cociente, notas="; ".join(notas)))


ver = pd.DataFrame({tic: veredicto_real(tic) for tic in sorted(ind["TIC"].unique())}).T
ver.index.name = "TIC"
ver = ver.reset_index()
ver.insert(1, "indice", ver["TIC"].map(lambda t: INDICE.get(t, np.nan)))
ver = ver.sort_values("indice")
ver.to_csv(DIR_FIG / f"veredicto_real_vs_sistematico_{FLUJO_TGLC}.csv", index=False)
pd.set_option("display.max_colwidth", 120)
ver

### 8.3 Figuras: profundidad de cada eclipse

Por sistema, una fila. **Izquierda:** profundidad de cada eclipse primario en el tiempo; los sectores
están uno al lado del otro (el eje x es días desde el inicio de cada sector). ● propia, × TGLC; las
líneas horizontales son las medianas por sector. **Derecha:** profundidad menos la mediana de su sector
contra el nivel de brillo fuera de eclipse (manchas → pendiente negativa).

In [ ]:
def figura_individuales(tic, axs):
    g = ind.loc[(ind["TIC"] == tic) & (ind["eclipse"] == "primario")]
    secs = sorted(g["Sector"].unique())
    x0 = {s: k * 32 for k, s in enumerate(secs)}
    for fot, mk, alfa in (("propia", "o", 0.8), ("tglc", "x", 0.9)):
        gg = g.loc[g["fotometria"] == fot]
        for s in secs:
            h = gg.loc[gg["Sector"] == s]
            if h.empty:
                continue
            col = COLOR_SECTOR.get(int(s), "0.4")
            xx = h["t"] - g.loc[g["Sector"] == s, "t"].min() + x0[s]
            axs[0].errorbar(xx, h["D"], yerr=h["err"], fmt=mk, color=col if fot == "propia" else "k",
                            ms=4, alpha=alfa, elinewidth=0.6, capsize=0, mfc=col if fot == "propia" else "none")
            axs[0].hlines(h["D"].median(), x0[s], x0[s] + 28, color=col if fot == "propia" else "k",
                          ls="-" if fot == "propia" else "--", lw=1.4)
            axs[1].plot(h["nivel"] - h["nivel"].mean(), h["D"] - h["D"].median(), mk,
                        color=col if fot == "propia" else "k", ms=4, alpha=alfa,
                        mfc=col if fot == "propia" else "none")
    axs[0].set_xticks([x0[s] + 14 for s in secs])
    axs[0].set_xticklabels([f"S{s}" for s in secs])
    axs[0].set_ylabel("D eclipse primario")
    axs[1].axhline(0, color="0.6", lw=0.8)
    axs[1].set_xlabel("brillo fuera de eclipse − media del sector")
    axs[1].set_ylabel("D − mediana del sector")
    v = ver.loc[ver["TIC"] == tic]
    idx = INDICE.get(tic, np.nan)
    axs[0].set_title((f"sistema {int(idx)} · " if np.isfinite(idx) else "") + f"TIC {tic}", loc="left", fontsize=10)
    if len(v):
        axs[1].set_title(v["veredicto"].iloc[0], loc="left", fontsize=8, color="#c0392b")
    for ax in axs:
        ax.grid(alpha=0.2)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)


orden = ver["TIC"].tolist()
fig, axs = plt.subplots(len(orden), 2, figsize=(15, 3.3 * len(orden)), squeeze=False,
                        gridspec_kw={"width_ratios": [2.2, 1], "hspace": 0.6, "wspace": 0.2})
for fila_ax, tic in zip(axs, orden):
    figura_individuales(tic, fila_ax)
fig.subplots_adjust(top=1 - 0.8 / fig.get_figheight())
fig.suptitle(f"Profundidad de cada eclipse primario: ● propia, × TGLC ({FLUJO_TGLC})",
             y=1 - 0.3 / fig.get_figheight(), fontsize=13)
fig.savefig(DIR_FIG / f"eclipses_individuales_{FLUJO_TGLC}.png", dpi=110, bbox_inches="tight")
plt.show()